# 04 — K-Means Clustering: Job Segmentation

**Tujuan:** Mengelompokkan job posting ke dalam cluster berdasarkan pola skill, untuk:
- Memahami struktur pasar IT (cluster mana yang dominan)
- Memberikan konteks tambahan pada user ("job ini termasuk Cloud & DevOps cluster")
- Diversifikasi rekomendasi (pastikan top-N tidak semua dari cluster yang sama)

**Algoritma:** K-Means dengan jumlah cluster ditentukan via **Elbow Method** + **Silhouette Score**.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from src.data_loader import load_jobs_data
from src.feature_engineering import build_skill_vocabulary, build_job_skill_matrix
from src.clustering import find_optimal_k, fit_kmeans, label_clusters

sns.set_style('whitegrid')
LI_BLUE = '#0A66C2'

In [ ]:
df, _ = load_jobs_data()
vocab = build_skill_vocabulary(df, skill_col='skills_list', min_freq=5)
job_matrix = build_job_skill_matrix(df, vocab, skill_col='skills_list')
print(f'Matrix shape: {job_matrix.shape}')

## 4.1 Find Optimal K — Elbow Method

In [ ]:
result = find_optimal_k(job_matrix, k_range=range(2, 11))
print(f'Optimal K (by max silhouette): {result["optimal_k"]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(result['k_values'], result['inertias'], 'o-', color=LI_BLUE, linewidth=2, markersize=8)
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.3)

# Silhouette plot
axes[1].plot(result['k_values'], result['silhouette_scores'], 'o-', color='#057642', linewidth=2, markersize=8)
axes[1].axvline(result['optimal_k'], color='red', linestyle='--', label=f'Optimal K={result["optimal_k"]}')
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette Score per K')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4.2 Fit K-Means with Chosen K

In [ ]:
# Pakai K=6 untuk granularity yang baik (atau gunakan optimal_k)
n_clusters = max(5, result['optimal_k'])
print(f'Using K = {n_clusters}')

kmeans, labels, scaler = fit_kmeans(job_matrix, n_clusters=n_clusters, scale=False)
df['cluster'] = labels

# Cluster size distribution
cluster_counts = pd.Series(labels).value_counts().sort_index()
print('\nCluster sizes:')
for cid, count in cluster_counts.items():
    print(f'  Cluster {cid}: {count:,} jobs ({count/len(df)*100:.1f}%)')

## 4.3 Label Clusters Based on Top Skills

In [ ]:
cluster_labels = label_clusters(kmeans, vocab, top_skills_per_cluster=8)

for cid, info in cluster_labels.items():
    print(f'\n=== Cluster {cid}: {info["label"]} ===')
    print(f'Top skills: {[s.title() for s in info["top_skills"]]}')
    # Top job titles in this cluster
    titles_in_cluster = df[df['cluster'] == cid]['title'].value_counts().head(5)
    print('Top job titles:')
    for t, n in titles_in_cluster.items():
        print(f'  {t} ({n})')

## 4.4 Visualize Clusters with PCA (2D Projection)

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(job_matrix)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))
for cid in range(n_clusters):
    mask = labels == cid
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=[colors[cid]], label=cluster_labels[cid]['label'],
               alpha=0.5, s=20)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Job Clusters (PCA 2D Projection)')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## 4.5 Cluster Profile Heatmap

In [ ]:
# Untuk tiap cluster, hitung % job yang punya tiap skill (top 20 skill)
from collections import Counter
from src.feature_engineering import parse_skills_list, normalize_skill_name

all_skills = []
for skills in df['skills_list']:
    parsed = parse_skills_list(skills)
    all_skills.extend([normalize_skill_name(s) for s in parsed if s])

top_20_skills = [s for s, _ in Counter(all_skills).most_common(20)]

heatmap_data = []
for cid in range(n_clusters):
    row = {}
    cluster_df = df[df['cluster'] == cid]
    n = len(cluster_df)
    for skill in top_20_skills:
        skill_idx = vocab.index(skill) if skill in vocab else None
        if skill_idx is not None:
            pct = job_matrix[df['cluster'] == cid, skill_idx].mean() * 100
            row[skill.title()] = pct
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data,
                          index=[cluster_labels[c]['label'] for c in range(n_clusters)])

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_df, annot=True, fmt='.0f', cmap='Blues', cbar_kws={'label': '% of jobs in cluster'}, ax=ax)
ax.set_title('Cluster Profile: % of jobs requiring each skill')
plt.tight_layout()
plt.show()

## Kesimpulan Clustering

- Pasar job IT secara natural terbagi ke beberapa cluster yang sangat distinct: data/ML, cloud/DevOps, frontend, backend, dst.
- Heatmap memperlihatkan setiap cluster punya skill signature yang khas
- Cluster ini bisa dipakai untuk:
  1. Diversifikasi rekomendasi (pastikan top-N tidak monoton dari satu cluster)
  2. Memberikan konteks tambahan pada user ("job ini di domain XYZ")
  3. Discover cross-cluster opportunity (user data scientist mungkin juga cocok dengan ML engineering cluster)